# Tutorial: Inference Replay Demo

Audience:
- Recruiters, hiring managers, and engineers reviewing the public demo.

Prerequisites:
- Install `requirements.txt`.
- Ensure `mondrian_artifacts_demo/` and the demo dataset CSV are present.

Learning goals:
- Load the saved demo bundle directly from disk.
- Generate a historical replay forecast without starting the API.
- Inspect interval provenance and basic backtest metrics.
- Simulate runtime recalibration on the trailing days before a chosen replay cutoff.


## Outline

1. Resolve the project root and load the replay service objects.
2. Inspect the public demo scope and allowed replay window.
3. Generate a forecast replay for one station.
4. Plot predictions, intervals, and realized values.
5. Run a small backtest summary.
6. Compare base vs recalibrated intervals on the same replay window.


In [ ]:
from __future__ import annotations

import os
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go


DATASET_CANDIDATES = ("df_stationsv3.csv", "df_stations_v3.csv")


def resolve_project2_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for base in candidates:
        if base.name == "project2" and any((base / name).exists() for name in DATASET_CANDIDATES):
            return base
        nested = base / "project2"
        if any((nested / name).exists() for name in DATASET_CANDIDATES):
            return nested
    raise FileNotFoundError("Could not locate project2 root from the current working directory.")


PROJECT2_ROOT = resolve_project2_root()
if str(PROJECT2_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT2_ROOT))
os.chdir(PROJECT2_ROOT)

from app.config import load_config
from app.models.calibration import CalibrationSettings, ManualCalibrationEngine
from app.models.bundle import load_demo_bundle
from app.models.data_store import DemoDataStore
from app.models.intervals import IntervalResolver
from app.models.service import DemoForecastService

cfg = load_config()
bundle = load_demo_bundle(cfg.artifact_path)
data_store = DemoDataStore(cfg)
intervals = IntervalResolver(bundle=bundle, runtime_states_dir=cfg.runtime_states_dir)
service = DemoForecastService(cfg=cfg, bundle=bundle, data_store=data_store, intervals=intervals)
engine = ManualCalibrationEngine(
    cfg=cfg,
    bundle=bundle,
    data_store=data_store,
    intervals=intervals,
    settings=CalibrationSettings(horizon=cfg.horizon, stride=4, beta=0.90, min_sht_n=6, min_sh_n=100),
)

status = service.status()
pd.Series(
    {
        "project_root": str(PROJECT2_ROOT),
        "artifact_path": status["artifact_path"],
        "dataset_start": status["dataset_start"],
        "dataset_end": status["dataset_end"],
        "allowed_forecast_start_min": status["allowed_forecast_start_min"],
        "allowed_forecast_start_max": status["allowed_forecast_start_max"],
        "num_stations": status["num_stations"],
        "horizon": status["horizon"],
    }
)


## Demo Scope

The public service is intentionally historical replay only. The same service objects used by the FastAPI app can be called directly from a notebook, which makes this walkthrough easier to run locally and easier to review on GitHub.


In [ ]:
stations = [s for s in cfg.active_stations if s not in cfg.excluded_stations]
station = stations[0]
forecast_start = data_store.default_forecast_start()
history_hours = 48

scope_df = pd.DataFrame(
    {
        "station": stations,
        "min_start": [service.forecast_start_bounds(s)[0] for s in stations],
        "max_start": [service.forecast_start_bounds(s)[1] for s in stations],
    }
)
scope_df


## Forecast Replay

Generate one replay forecast using the saved model artifact, station scaler, and Mondrian interval resolver. Because we stay inside the historical dataset window, we can plot the realized values against the interval.


In [ ]:
forecast = service.forecast(
    station=station,
    forecast_start=forecast_start,
    history_hours=history_hours,
)

summary = pd.Series(
    {
        "station": forecast["station"],
        "forecast_start": forecast["forecast_start"],
        "history_points": len(forecast["historical_values"]),
        "horizon": forecast["horizon"],
        "interval_sources": dict(Counter(forecast["_artifacts"]["interval_sources"])),
    }
)
summary


In [ ]:
hist_df = pd.DataFrame(
    {
        "timestamp": pd.to_datetime(forecast["historical_timestamps"], utc=True),
        "history": forecast["historical_values"],
    }
)
fc_df = pd.DataFrame(
    {
        "timestamp": pd.to_datetime(forecast["timestamps"], utc=True),
        "prediction": forecast["predictions"],
        "lower": forecast["lower_bound"],
        "upper": forecast["upper_bound"],
        "actual": forecast["actual_values"],
    }
)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=hist_df["timestamp"],
        y=hist_df["history"],
        mode="lines",
        name="History",
        line=dict(color="#486581", width=2),
    )
)
fig.add_trace(
    go.Scatter(
        x=list(fc_df["timestamp"]) + list(fc_df["timestamp"][::-1]),
        y=list(fc_df["upper"]) + list(fc_df["lower"][::-1]),
        fill="toself",
        fillcolor="rgba(15, 76, 129, 0.18)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        name="Prediction interval",
    )
)
fig.add_trace(
    go.Scatter(
        x=fc_df["timestamp"],
        y=fc_df["prediction"],
        mode="lines",
        name="Prediction",
        line=dict(color="#0f4c81", width=3),
    )
)
fig.add_trace(
    go.Scatter(
        x=fc_df["timestamp"],
        y=fc_df["actual"],
        mode="lines",
        name="Actual",
        line=dict(color="#d1495b", width=2),
    )
)
fig.update_layout(
    title=f"Historical replay forecast for {station}",
    xaxis_title="Timestamp (UTC)",
    yaxis_title="Flow / consumption",
    template="plotly_white",
    height=520,
)
fig


## Backtest Snapshot

The replay service can also evaluate the same historical window as a backtest. This is useful for communicating that the public demo is about calibrated interval behavior, not just producing a pretty forecast line.


In [ ]:
backtest = service.backtest(
    station=station,
    backtest_start=forecast_start,
    history_hours=history_hours,
)

pd.Series(
    {
        "coverage": backtest["coverage"],
        "mae": backtest["metrics"]["mae"],
        "rmse": backtest["metrics"]["rmse"],
        "mean_interval_width": backtest["metrics"]["mean_interval_width"],
        "n_valid_points": backtest["metrics"]["n_valid_points"],
        "n_total_points": backtest["metrics"]["n_total_points"],
    }
)


## Replay Calibration Simulation

The public demo can simulate a runtime recalibration step by using only the trailing `N` days before the chosen replay start. This keeps the calibration causal: the overlay is fit using only information available before the forecast being inspected.


In [ ]:
calibration_days = 14

intervals.clear_runtime_state(station)
before_cal = service.forecast(
    station=station,
    forecast_start=forecast_start,
    history_hours=history_hours,
    use_runtime_overlays=False,
)
calibration_result = engine.calibrate_station(
    station=station,
    days=calibration_days,
    reference_time=forecast_start,
)
after_cal = service.forecast(
    station=station,
    forecast_start=forecast_start,
    history_hours=history_hours,
    use_runtime_overlays=True,
)

pd.Series(
    {
        "calibration_days": calibration_days,
        "reference_time": calibration_result["reference_time"],
        "updated_keys": calibration_result["updated_keys"],
        "empirical_coverage_during_calibration": calibration_result["empirical_coverage"],
    }
)


## Variations

- Change `station` to compare different regimes.
- Change `calibration_days` and rerun the replay calibration cell to see how interval widths adapt.
- Move `forecast_start` within the allowed window to inspect interval behavior across time.
- Call `intervals.clear_runtime_state(station)` to reset back to the immutable base artifact.
- Run the FastAPI app if you want the same replay flow through HTTP and the dashboard UI.
